# SETU-RAG — Colab T4 walkthrough

Code-Switch-Aware Multilingual RAG for 22 Indian languages (customer support).
Run top-to-bottom on a **T4** runtime.

## 1 · Get the code & install deps

In [ ]:
import os
if not os.path.isdir('setu-rag'):
    !git clone https://github.com/Gaurs86/setu-rag.git
os.chdir('setu-rag')
print('cwd:', os.getcwd())
!pip -q install -r colab_requirements.txt
# Optional: bring the full AI4Bharat front-end alive (each is real-with-fallback):
# !pip -q install ai4bharat-transliteration IndicTransToolkit
# !pip -q install git+https://github.com/AI4Bharat/IndicLID.git

## 2 · Front-end: LID + CMI on a Hinglish query (no GPU needed)

In [ ]:
from setu_rag.front_end.language_id import LanguageIdentifier
from setu_rag.front_end.cmi import compute_cmi
lid = LanguageIdentifier().load()
q = 'mera refund kab tak aayega, order cancel kar diya tha'
toks = lid.tag(q)
print(compute_cmi(toks))

## 3 · Adaptive routing by CMI  [novel #1]

In [ ]:
from setu_rag.router.adaptive_router import decide_route
cmi = compute_cmi(toks)
frac_roman = sum(t.script=='Latn' for t in toks)/len(toks)
print(decide_route(cmi, frac_roman))

## 4 · Build the index and run the full pipeline (real models)

In [ ]:
import sys; sys.path.insert(0, os.getcwd())
from setu_rag.app import build_pipeline

rag = build_pipeline()   # downloads ~3-4 GB weights on first run; uses fallbacks on CPU

for q in ['mera refund kab tak aayega, order cancel kar diya tha',
          'how do I track my order',
          'coupon apply nahi ho raha hai']:
    tr = rag.answer(q)
    print(f'\nQ: {q}')
    print(f'   route={tr.route}  cmi={tr.cmi:.2f}  matrix={tr.matrix_lang}  grade={tr.grade}')
    print(f'   A: {tr.answer}')

## 5 · CMI-conditioned prompt preview  [novel #3]

In [ ]:
from setu_rag.generation.generator import build_prompt
ctx = [{'answer':'Refund 5-7 business days mein aa jata hai.'}]
print(build_prompt(q, ctx, cmi.matrix_lang, cmi.cmi))

## 6 · CS-RAGAS metrics

In [ ]:
from setu_rag.eval import cs_metrics
print('CMI-alignment:', cs_metrics.cmi_alignment(q, 'aapka refund 5-7 din mein aa jayega', lid))